# Step 1
### Analyze and extract the json information into three parquet files 

![](../docs/images/analyze_and_extract.png)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, explode, explode_outer, split
from pyspark.sql.types import StringType, IntegerType, StructType, StructField
from pyspark.sql.functions import col
from pyspark.sql.types import ArrayType,StructType
import json

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("Read JSON Data") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
# Read JSON file from S3 bucket
file = "s3a://evdata-test/raw/ElectricVehiclePopulationData.json"
multiline_df = spark.read.option("multiline", "true") \
      .json(file)
multiline_df.printSchema()
multiline_df.show()

In [ ]:
# Generate 3 data frames one each for table metadata, column metadata and vehicle data
table_metadata = multiline_df.select("meta.view.*").drop("columns")
columns_metadata = multiline_df.select(explode(col("meta.view.columns")).alias("columns"))
columns_metadata = columns_metadata.select("columns.*")
vehicle_data = multiline_df.select(explode(col("data")).alias("row_data"))

In [ ]:
# Function to flatten the json by iterating through fields - both arrays and structs


def flatten_json(df):

    """
    Flattens a DataFrame with complex nested fields (Arrays and Structs) by converting them into individual columns.
   
    Parameters:
    - df: The input DataFrame with complex nested fields
   
    Returns:
    - The flattened DataFrame with all complex fields expanded into separate columns.
   """
   # compute Complex Fields (Lists and Structs) in Schema   
    complex_fields = dict([(field.name, field.dataType)
                             for field in df.schema.fields
                             if type(field.dataType) == ArrayType or  type(field.dataType) == StructType])
    print(df.schema)
    print("")
    while len(complex_fields)!=0:
      col_name=list(complex_fields.keys())[0]
      print ("Processing :"+col_name+" Type : "+str(type(complex_fields[col_name])))
    
      # if StructType then convert all sub element to columns.
      # i.e. flatten structs
      if (type(complex_fields[col_name]) == StructType):
         expanded = [col(col_name+'.'+k).alias(col_name+'_'+k) for k in [ n.name for n in  complex_fields[col_name]]]
         df=df.select("*", *expanded).drop(col_name)
    
      # if ArrayType then add the Array Elements as Rows using the explode function
      # i.e. explode Arrays
      elif (type(complex_fields[col_name]) == ArrayType):    
         df=df.withColumn(col_name,explode_outer(col_name))
    
      # recompute remaining Complex Fields in Schema       
      complex_fields = dict([(field.name, field.dataType)
                             for field in df.schema.fields
                             if type(field.dataType) == ArrayType or  type(field.dataType) == StructType])
    return df

In [ ]:
# Flatten table metadata and exclude metadata fields that has special characters
table_metadata = multiline_df.select("meta.view.*").drop("columns")
table_meta = table_metadata.select("*").drop("metadata")
flatten_table_meta_df = flatten_json(table_meta)
flatten_table_meta_df.display()

In [ ]:
columns_metadata.display()

In [ ]:
vehicle_data.display()

In [ ]:
# AWS credentials are read from environment variables / Databricks secrets.
# Never hardcode credentials in code.
import os

spark.conf.set("fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
spark.conf.set("fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
spark.conf.set("fs.s3a.endpoint", "s3.amazonaws.com")


In [ ]:
flatten_table_meta_df.coalesce(1).write.mode("overwrite").parquet("s3a://evdata-test/derived/table_metadata")
columns_metadata.coalesce(1).write.mode("overwrite").parquet("s3a://evdata-test/derived/columns_metadata")



In [ ]:
data_headers = columns_metadata.select("name").rdd.flatMap(lambda x: x).collect()
vehicle_data_exploded = vehicle_data.select(*[col('row_data').getItem(i).alias(f'row_data{i+1}') for i in range(0, 28)])
vehicle_data_exploded = vehicle_data_exploded.toDF(*data_headers)
#vehicle_data_exploded.display()

vehicle_data_exploded.write.mode("overwrite").parquet("s3a://evdata-test/derived/vehicle_data")



# Findings 

JSON has two elements 
1. meta
2. data 

meta consists of metadata information that has data set metadata - referring this as table metadata going forward and column metadata. 

table metadata consists of various types of arrays and struct fields of which approvals, submission information is also part of it. Table metadata is flattened before being written to S3 bucket

column metadata consists of name, data type, description, position and other details. 

names from column metadata are extracted as first row and stitched together with vehicle data and written as a parquet file to the derived folder

![](/Volumes/de_demo/default/ev_data/S3 File processing.png)